# Hospital and facility inspection

This notebook starts by extracting the facility name from every encounter using:

- `encounter_fhir.encounter.serviceProvider.display` — organization/facility name
- `encounter_fhir.encounter.location[].location.display` — encounter location name
- `encounter_fhir.encounter.participant[].individual.display` — recorded practitioner

> The source field contains all kinds of care facilities—not only hospitals. The 25 encounters include outpatient clinics, hospitals, skilled-nursing facilities, and hospice providers. The data is synthetic and facility assignments may not perfectly match the clinical scenario.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

DATASET_NAME = "synthetic-ambient-fhir-25.jsonl"
DATASET_CANDIDATES = [
    Path("data/synthetic-ambient-fhir-25") / DATASET_NAME,
    Path("../data/synthetic-ambient-fhir-25") / DATASET_NAME,
]

DATASET_PATH = next((path for path in DATASET_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    checked = "\n".join(f"- {path.resolve()}" for path in DATASET_CANDIDATES)
    raise FileNotFoundError(f"Dataset not found. Checked:\n{checked}")

with DATASET_PATH.open(encoding="utf-8") as handle:
    records = [json.loads(line) for line in handle if line.strip()]

assert len(records) == 25, f"Expected 25 encounters, found {len(records)}"
print(f"Loaded {len(records)} encounters from {DATASET_PATH.resolve()}")

Loaded 25 encounters from /Users/robbertstruyven/Dropbox/Mac/Downloads/abridge_hackathon/abridge-hackathon/data/synthetic-ambient-fhir-25/synthetic-ambient-fhir-25.jsonl


## Extract facility names from all encounters

The table below preserves the organization and location display names exactly as supplied by FHIR.

In [2]:
def patient_name(record):
    names = record["patient_context"]["patient"].get("name", [])
    if not names:
        return "Unknown"
    name = names[0]
    return " ".join([*name.get("given", []), name.get("family", "")]).strip()


def reference_identifier(reference):
    """Return the identifier portion of a conditional FHIR reference."""
    if not reference:
        return None
    return reference.split("|", 1)[-1] if "|" in reference else reference


encounter_rows = []
for encounter_number, record in enumerate(records):
    encounter = record["encounter_fhir"]["encounter"]
    service_provider = encounter.get("serviceProvider", {})
    locations = encounter.get("location", [])
    participants = encounter.get("participant", [])

    location = locations[0].get("location", {}) if locations else {}
    practitioner = participants[0].get("individual", {}) if participants else {}
    period = encounter.get("period", {})

    encounter_rows.append({
        "encounter_number": encounter_number,
        "patient": patient_name(record),
        "encounter_date": record["metadata"]["date"][:10],
        "visit_title": record["metadata"]["visit_title"],
        "encounter_class": encounter.get("class", {}).get("code"),
        "facility_name": service_provider.get("display"),
        "facility_identifier": reference_identifier(service_provider.get("reference")),
        "location_name": location.get("display"),
        "location_identifier": reference_identifier(location.get("reference")),
        "practitioner_name": practitioner.get("display"),
        "practitioner_reference": practitioner.get("reference"),
        "period_start": period.get("start"),
        "period_end": period.get("end"),
    })

encounters_df = pd.DataFrame(encounter_rows)
display(
    encounters_df[
        [
            "encounter_number",
            "patient",
            "encounter_date",
            "encounter_class",
            "facility_name",
            "location_name",
            "visit_title",
        ]
    ]
)

,encounter_number,patient,encounter_date,encounter_class,facility_name,location_name,visit_title
0,0,Ali Jerry Kuhic,2022-08-05,AMB,WHITLEY WELLNESS LLC,WHITLEY WELLNESS LLC,Annual physical — preventive screening and migraine check-in
1,1,Ariane Jan Runolfsson,2021-01-03,IMP,"FIRST PSYCHIATRIC PLANNERS, INC","FIRST PSYCHIATRIC PLANNERS, INC",Inpatient admission — COVID-19 isolation with pneumonia and hypoxemia
2,2,Clarence Reinger,2025-06-01,AMB,CAMBRIDGE PUBLIC HEALTH COMMISSION,CAMBRIDGE PUBLIC HEALTH COMMISSION,Prenatal intake visit — initial obstetric evaluation
3,3,Corinne Charlesetta Crooks,2017-01-01,AMB,"LEAP INTO WELLNESS,LLC","LEAP INTO WELLNESS,LLC",Annual wellness examination — preventive screening and health maintenance
4,4,Delana Jamie Gutkowski,2025-05-23,AMB,MEASURED WELLNESS LLC,MEASURED WELLNESS LLC,Young adult preventive exam — prediabetes and allergy follow-up
5,5,Dick Larson,2024-05-18,AMB,CHILDREN'S HOSPITAL CORPORATION,CHILDREN'S HOSPITAL CORPORATION,Annual check-up — post-sepsis recovery and prediabetes
6,6,Elias Wisozk,2026-06-18,AMB,GREATER LAWRENCE FAMILY HEALTH CENTER INC,GREATER LAWRENCE FAMILY HEALTH CENTER INC,General adult exam — new hypertension and metabolic syndrome
7,7,Emory Britt Kovacek,2021-04-06,AMB,SUNRISE HEALTHCARE LLC,SUNRISE HEALTHCARE LLC,General exam — chronic low back pain and positive depression screen
8,8,Eva Julia Casas,2016-08-30,AMB,"PRIMARY CARE MEDICINE AND PEDIATRICS, LLC","PRIMARY CARE MEDICINE AND PEDIATRICS, LLC","Annual general exam — prediabetes, hyperlipidemia, and knee osteoarthritis"
9,9,Hiedi Moira Thiel,2024-07-22,AMB,ENCOMPASS HEALTH REHAB HOSPITAL OF WESTERN MASS,ENCOMPASS HEALTH REHAB HOSPITAL OF WESTERN MASS,"Prenatal intake visit — first trimester, newly identified anemia"


## Unique facilities

A facility can occur in more than one encounter. This summary deduplicates by the source organization identifier rather than by name alone.

In [3]:
facility_summary_df = (
    encounters_df.groupby(["facility_identifier", "facility_name"], dropna=False)
    .agg(
        encounter_count=("encounter_number", "count"),
        patient_count=("patient", "nunique"),
        encounter_classes=("encounter_class", lambda values: ", ".join(sorted(set(values.dropna())))),
        first_encounter=("encounter_date", "min"),
        last_encounter=("encounter_date", "max"),
    )
    .reset_index()
    .sort_values(["encounter_count", "facility_name"], ascending=[False, True])
    .reset_index(drop=True)
)

display(facility_summary_df)
print(f"{len(facility_summary_df)} unique facilities across {len(encounters_df)} encounters")

,facility_identifier,facility_name,encounter_count,patient_count,encounter_classes,first_encounter,last_encounter
0,a2d45ee8-b930-3169-b612-f602961ad212,SUNRISE HEALTHCARE LLC,2,2,AMB,2021-04-06,2024-10-07
1,2f735290-1603-31b5-b561-7c8e0d413170,BL HEALTHCARE INC DELEWARE,1,1,AMB,2025-07-13,2025-07-13
2,bac9dc0c-cb0f-3fa6-aa12-819dd0fb922a,CAMBRIDGE PUBLIC HEALTH COMMISSION,1,1,AMB,2025-06-01,2025-06-01
3,e6cc6043-299a-3581-8f6b-b3b5d62c46ae,"CARING HEALTH CENTER, INC",1,1,AMB,2020-06-17,2020-06-17
4,a2cc09de-c1e2-3126-8c09-d49f95225281,CHILDREN'S HOSPITAL CORPORATION,1,1,AMB,2024-05-18,2024-05-18
5,9a20d707-68e4-32c0-b2b2-2f5d5c2012cd,CLARITY HEALTH & WELLNESS LLC,1,1,AMB,2019-07-14,2019-07-14
6,11ba6a26-59b4-3728-a268-6dc9929962b4,COURTYARD NURSING CARE CENTER,1,1,IMP,2023-11-27,2023-11-27
7,0fab3c4e-a028-39a6-b15a-be5a88b6aa2a,ENCOMPASS HEALTH REHAB HOSPITAL OF WESTERN MASS,1,1,AMB,2024-07-22,2024-07-22
8,5018c664-e283-30eb-932a-529d9a19b3b5,"FIRST PSYCHIATRIC PLANNERS, INC",1,1,IMP,2021-01-03,2021-01-03
9,5581e43d-0463-3a0b-afc7-a60aea5ac96c,GREATER LAWRENCE FAMILY HEALTH CENTER INC,1,1,AMB,2026-06-18,2026-06-18


24 unique facilities across 25 encounters


## Data completeness and consistency checks

In [4]:
checks = pd.Series({
    "encounters": len(encounters_df),
    "missing facility names": encounters_df["facility_name"].isna().sum(),
    "missing facility identifiers": encounters_df["facility_identifier"].isna().sum(),
    "missing location names": encounters_df["location_name"].isna().sum(),
    "facility/location name mismatches": (
        encounters_df["facility_name"] != encounters_df["location_name"]
    ).sum(),
    "missing practitioner names": encounters_df["practitioner_name"].isna().sum(),
})
display(checks.rename("count").to_frame())

display(encounters_df["encounter_class"].value_counts(dropna=False).rename("encounters").to_frame())

,count
encounters,25
missing facility names,0
missing facility identifiers,0
missing location names,0
facility/location name mismatches,0
missing practitioner names,0


,encounters
encounter_class,
AMB,19
IMP,4
HH,2


## Inspect one encounter

Change `ENCOUNTER_NUMBER` from 0 to 24 to inspect the raw facility, location, practitioner, and encounter information.

In [5]:
ENCOUNTER_NUMBER = 0

selected_record = records[ENCOUNTER_NUMBER]
selected_encounter = selected_record["encounter_fhir"]["encounter"]

display(encounters_df[encounters_df["encounter_number"] == ENCOUNTER_NUMBER].T)
print("\nRaw Encounter resource:")
display(selected_encounter)

# Optional local exports (uncomment when needed):
# output_dir = Path("data_processing/outputs")
# output_dir.mkdir(parents=True, exist_ok=True)
# encounters_df.to_csv(output_dir / "encounter_facilities.csv", index=False)
# facility_summary_df.to_csv(output_dir / "unique_facilities.csv", index=False)

,0
encounter_number,0
patient,Ali Jerry Kuhic
encounter_date,2022-08-05
visit_title,Annual physical — preventive screening and migraine check-in
encounter_class,AMB
facility_name,WHITLEY WELLNESS LLC
facility_identifier,e2a8b444-9b8f-36ff-84c4-05ee98589482
location_name,WHITLEY WELLNESS LLC
location_identifier,16162e2e-393c-3162-8e09-a84a60d825c4
practitioner_name,Dr. Francisco472 Gusikowski974



Raw Encounter resource:


{'resourceType': 'Encounter',
 'id': '3a3a1f26-ed23-f65c-e264-be689558faea',
 'meta': {'profile': ['http://hl7.org/fhir/us/core/StructureDefinition/us-core-encounter']},
 'identifier': [{'use': 'official',
   'system': 'https://github.com/synthetichealth/synthea',
   'value': '3a3a1f26-ed23-f65c-e264-be689558faea'}],
 'status': 'finished',
 'class': {'system': 'http://terminology.hl7.org/CodeSystem/v3-ActCode',
  'code': 'AMB'},
 'type': [{'coding': [{'system': 'http://snomed.info/sct',
     'code': '162673000',
     'display': 'General examination of patient (procedure)'}],
   'text': 'General examination of patient (procedure)'}],
 'subject': {'reference': 'urn:uuid:3a3a1f26-ed23-f65c-a7df-c96fac56f464',
  'display': 'Mr. Ali918 Jerry654 Kuhic920'},
 'participant': [{'type': [{'coding': [{'system': 'http://terminology.hl7.org/CodeSystem/v3-ParticipationType',
       'code': 'PPRF',
       'display': 'primary performer'}],
     'text': 'primary performer'}],
   'period': {'start': '20